# WaveDL GUI — quick Colab test

Run cells **1–3**, then open the GUI link to configure and start training yourself. Optionally select **Runtime → Change runtime type → T4 GPU** before starting. CPU is enough for this smoke test.

This clones `lndmanh/WaveDL-GUI`, branch `review/daemon-browser-ci-rebuild`, and installs its bundled GUI in an isolated Python environment. **Only pushed commits are included**, not uncommitted local changes. Dependencies must be available for Colab's Python/platform; installation failures stop setup rather than silently bypassing version requirements.

Use the bundled GUI for the quickest start. Enable **REBUILD_GUI** to build it from the branch's web source instead (requires Node.js 22.19+ in the runtime). Data and runs live under `/content/wavedl-gui-test` and disappear when Colab deletes the runtime. Keep bootstrap links private and clear cell outputs before sharing this notebook.

In [ ]:
# 1. Clone this branch and install the package.
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/lndmanh/WaveDL-GUI.git' #@param {type: 'string'}
BRANCH = 'review/daemon-browser-ci-rebuild' #@param {type: 'string'}
REBUILD_GUI = False #@param {type: 'boolean'}

ROOT = Path('/content/wavedl-gui-test')
ROOT.mkdir(mode=0o700, parents=True, exist_ok=True)
ROOT.chmod(0o700)
REPO = ROOT / 'checkout'
PYTHON = ROOT / 'venv' / 'bin' / 'python'
if REPO.exists():
    print('Reusing existing checkout; no reset or pull is performed on rerun.')
    print('For a fresh branch checkout, delete the Colab runtime and reconnect.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, '--', REPOSITORY, str(REPO)], check=True, cwd='/content')
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--format=%h %s'], check=True)
if not PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ROOT / 'venv')], check=True, cwd='/content')
if REBUILD_GUI:
    subprocess.run(['node', '--version'], check=True)
    subprocess.run(['npm', 'ci'], cwd=REPO / 'src/wavedl_daemon/web', check=True)
    subprocess.run(['npm', 'run', 'generate'], cwd=REPO / 'src/wavedl_daemon/web', check=True)
    subprocess.run([sys.executable, 'tools/vendor_web_assets.py'], cwd=REPO, check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True, cwd='/content')
subprocess.run([str(PYTHON), '-m', 'pip', 'install', f'{REPO}[daemon]'], check=True, cwd='/content')
subprocess.run([str(PYTHON), '-c', 'import torch; print("CUDA available:", torch.cuda.is_available())'], check=True, cwd='/content')
print(f'Training interpreter: {PYTHON}')

In [ ]:
# 2. Create a tiny synthetic dataset; training is started only from the GUI.
DATA = ROOT / 'datasets'
DATA.mkdir(mode=0o700, exist_ok=True)
subprocess.run([str(PYTHON), '-c', '''
import sys
from pathlib import Path
import numpy as np
path = Path(sys.argv[1]) / 'gui_smoke_train.npz'
if not path.exists():
    rng = np.random.default_rng(42)
    inputs = rng.random((128, 64, 64), dtype=np.float32)
    targets = np.stack([inputs.mean(axis=(1, 2)), inputs.std(axis=(1, 2)), inputs.max(axis=(1, 2))], axis=1)
    np.savez(path, input_train=inputs, output_train=targets)
print(path)
''', str(DATA)], check=True, cwd='/content')
print('128 samples, 64 × 64 inputs, 3 regression targets. Workflow test, not a benchmark.')

In [ ]:
# 3. Start the daemon and open its GUI through Colab's authenticated proxy.
import html
import os
import re
import time
from urllib.parse import urlsplit
from google.colab import output
from IPython.display import HTML, display

if 'daemon' in globals() and daemon.poll() is None:
    raise RuntimeError('GUI is already running. Use the stop cell before reopening for a new sign-in link.')
PORT = 8420
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
proxy = urlsplit(proxy_url)
if proxy.scheme != 'https' or not proxy.netloc or proxy.path not in ('', '/'):
    raise RuntimeError(f'Expected a dedicated HTTPS Colab proxy origin, got {proxy_url!r}.')
proxy_origin = f'https://{proxy.netloc}'

# Only this notebook launcher knows the exact Colab origin. The package is unchanged.
launcher = r'''
import secrets
import sys
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4
import uvicorn
from wavedl import __version__
from wavedl.core.live_push import LivePushConfig
from wavedl_daemon.backends.local import LocalBackend
from wavedl_daemon.http.app import create_app
from wavedl_daemon.http.bootstrap import BootstrapCredentials
from wavedl_daemon.runs.runtime_service import HostedServices
from wavedl_daemon.storage.settings import Settings

root, external_origin, port_text = sys.argv[1:]
port = int(port_text)
external_host = urlsplit(external_origin).netloc
local_host = f'127.0.0.1:{port}'
token = secrets.token_urlsafe(32)
bootstrap = BootstrapCredentials(token)
settings = Settings(Path(root) / 'state', Path(root) / 'runs')
live_push = LivePushConfig(f'http://{local_host}', token)
services = HostedServices(
    settings,
    backend_factory=lambda lock: {'local': LocalBackend(dispatch_lock=lock, live_push=live_push)},
    browse_roots={'datasets': Path(root) / 'datasets', 'runs': settings.run_root},
)
app = create_app(
    settings, token=token, daemon_id=str(uuid4()), build_id=__version__,
    services=services, allowed_hosts=(local_host, f'localhost:{port}', external_host),
    bootstrap_credentials=bootstrap,
)

class ColabProxy:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope['type'] == 'http':
            hosts = [v.decode('latin-1') for k, v in scope['headers'] if k.lower() == b'host']
            origins = [v.decode('latin-1') for k, v in scope['headers'] if k.lower() == b'origin']
            # Colab may preserve the public host or replace it with localhost.
            # Normalize only the exact, browser-provided Colab origin; foreign
            # origins and duplicate headers still reach the daemon's rejection.
            if len(hosts) == 1 and hosts[0] in {external_host, local_host, f'localhost:{port}'}:
                if origins == [external_origin]:
                    scope = dict(scope)
                    scope['scheme'] = 'https'
                    scope['headers'] = [(k, external_host.encode('ascii') if k.lower() == b'host' else v) for k, v in scope['headers']]
        await self.app(scope, receive, send)

class ReadyServer(uvicorn.Server):
    async def startup(self, sockets=None):
        await super().startup(sockets=sockets)
        if self.started:
            print('GUI_BOOTSTRAP=' + bootstrap.mint(), flush=True)

ReadyServer(uvicorn.Config(
    ColabProxy(app), host='127.0.0.1', port=port, lifespan='on',
    proxy_headers=False, access_log=False, ws='none',
    timeout_graceful_shutdown=10,
)).run()
'''
log_path = ROOT / 'daemon.log'
descriptor = os.open(log_path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
with os.fdopen(descriptor, 'w') as log:
    daemon = subprocess.Popen([str(PYTHON), '-u', '-c', launcher, str(ROOT), proxy_origin, str(PORT)], cwd='/content', stdout=log, stderr=subprocess.STDOUT)
deadline = time.monotonic() + 90
while time.monotonic() < deadline:
    text = log_path.read_text()
    match = re.search(r'^GUI_BOOTSTRAP=([A-Za-z0-9_-]+)$', text, re.MULTILINE)
    if daemon.poll() is not None:
        raise RuntimeError(f'Daemon exited. Inspect {log_path} for the startup error.')
    if match:
        gui_url = f'{proxy_origin}/?bootstrap={match.group(1)}'
        display(HTML(f'<a href="{html.escape(gui_url, quote=True)}" target="_blank" rel="noopener noreferrer">Open WaveDL GUI</a> — sign in within 120 seconds.'))
        print(f'Training interpreter: {PYTHON}')
        print(f'Dataset: {DATA / "gui_smoke_train.npz"}')
        break
    time.sleep(0.5)
else:
    daemon.terminate()
    daemon.wait(timeout=20)
    raise TimeoutError(f'Daemon did not become ready. Inspect {log_path}.')

## Try a training run in the GUI

Create a new training run using the **Local** runtime and the training interpreter printed above. Browse the `datasets` root and select `gui_smoke_train.npz`. Use **CNN**, **3 output targets**, batch size **16**, **2 epochs**, and **0 data-loader workers** for a quick smoke test. If dataset keys are requested, use `input_train` and `output_train`.

Review/preview the plan, approve it, then start training and inspect its progress and artifacts. No notebook cell starts training automatically. For your own data, upload it into `/content/wavedl-gui-test/datasets` using Colab's Files panel.

Keep the Colab runtime connected. Closing the GUI does not stop training. If the one-use sign-in link expires, stop the daemon below and rerun cell 3; this also interrupts active local jobs. This is a Colab convenience launcher, not a claim of production or GPU qualification.

In [ ]:
# Optional: stop the daemon before restarting or ending the test.
# Stop/cancel active training in the GUI first.
if 'daemon' in globals() and daemon.poll() is None:
    daemon.terminate()
    try:
        daemon.wait(timeout=30)
    except subprocess.TimeoutExpired:
        raise RuntimeError('Daemon is still shutting down; do not start another instance yet.') from None
    print('Daemon stopped. Files remain until the Colab runtime is deleted.')
else:
    print('No notebook-owned daemon is running.')